# Supervised cold start

**Notebook 03 · Learn from worked solutions**

Before the student learns from feedback on its own answers, we teach it from a
small set of worked examples. This creates the supervised starting point used by GKD.
The selected profile defaults to **Qwen3.5-0.8B-Base**. `legacy` selects the historical
Qwen3 student. Use the matching environment described in the [run guide](assets/05_qwen38_restart/RUN.md).

| Setting | Fixed choice |
| :--- | :--- |
| Practice / validation | 256 / 64 disjoint GSM8K train questions |
| Update | Full student fine-tuning, one epoch, 32 accumulated updates |
| Targets | Worked response plus its genuine end token |
| Precision | FP32 weights, BF16 autocast on supported GPUs |
| Evaluation | Same greedy settings before and after SFT |
| Held-out test | Deferred to the fixed final experiment in 04 |

The [saved original execution](https://github.com/igoldin670/on-policy-distillation/blob/34f5aa3af61f840c5edca550cfd4ff2ca2521ad7/notebooks/03_sft_cold_start.ipynb)
belongs to the older model. Current source outputs start empty. New-model results
become evidence only after running training on the allocated GPU.

![The experiment pipeline](assets/03_sft_cold_start/sft_workflow.png)

*This project moves from supervised preparation to fresh student responses and teacher feedback.*

## 1. Prepare the runtime

Use `MODE = "inspect"` to check actual data and token alignment on CPU. Choose
`"train"` only in the allocated GPU kernel. Running 01 or 02 first is unnecessary.
For Qwen3.5/Qwen3.8, activate `~/venvs/on-policy-qwen38`.

In [ ]:
import os
import sys
from pathlib import Path

# A remote Jupyter kernel uses files on the remote machine. Find that checkout
# before importing helpers. Uploading a project ZIP is optional.
PROJECT_ROOT = next(
    (
        p
        for p in (
            Path.cwd(),
            *Path.cwd().parents,
            Path.home() / "on-policy-distillation",
            Path("/content/on-policy-distillation"),
        )
        if (p / "src/pipeline.py").is_file()
    ),
    None,
)
assert PROJECT_ROOT is not None, "Sync or clone the full project before running this notebook."
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
# Set caches before importing Hugging Face. Amarel scratch is used only for cache.
scratch_directory = Path("/scratch") / os.environ.get("USER", "")
IN_AMAREL = bool(
    os.environ.get("SLURM_JOB_ID") and os.environ.get("USER") and scratch_directory.is_dir()
)
cache_directory = (
    scratch_directory / "huggingface" if IN_AMAREL else PROJECT_ROOT / ".cache/source-verification"
)
os.environ.setdefault("HF_HOME", str(cache_directory))
os.environ.setdefault("HF_HUB_CACHE", str(Path(os.environ["HF_HOME"]) / "hub"))
os.environ.setdefault("HF_DATASETS_CACHE", str(Path(os.environ["HF_HOME"]) / "datasets"))
cache_directory.mkdir(parents=True, exist_ok=True)
# Public pinned downloads do not need Colab's secret-vault integration.
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".cache/matplotlib"))
MODEL_PROFILE = os.environ.get("OPD_MODEL_PROFILE", "qwen38")
print("Project", PROJECT_ROOT)
print("Profile", MODEL_PROFILE)
print("Cache", os.environ["HF_HOME"])

MODE = os.environ.get("OPD_SFT_MODE", "inspect")
assert MODE in {"inspect", "train"}
RESUME_RUN_DIRECTORY = None  # Set the existing run folder here after an interruption.

## 2. Keep practice and validation separate

Practice examples supply the gradients. Validation questions measure what changed.
Both come from GSM8K's pinned training split. Previously reserved test questions,
duplicated question text, and overlong examples are excluded deterministically.

Calculator markup such as `<<3*5=15>>` is removed while the written calculation and
final `#### 15` line are retained. The example below is actual practice data.

In [ ]:
from dataclasses import asdict

import pandas as pd
from IPython.display import display

from src.pipeline import prepare_cold_start, run_cold_start

# This loads pinned tokenizer/configuration files and data, without model weights.
state = prepare_cold_start(PROJECT_ROOT, MODEL_PROFILE)
config = state["sft_config"]
tokenizer = state["tokenizer"]
train_examples = state["cold_splits"]["train"]
validation_examples = state["cold_splits"]["validation"]
print(asdict(config))
print("Practice / validation", len(train_examples), len(validation_examples))
print("Recorded exclusions", len(state["cold_splits"]["exclusions"]))
example = train_examples[0]
print("Question\n", example["question"])
print("Worked response\n", example["target_text"])

## 3. Which tokens teach the model?

A complete example contains a **prompt**, a **worked response**, and an **end token**.
`input_ids` contains all three. `labels` contains response IDs and the genuine end
ID, with `-100` at prompt and padding positions. PyTorch ignores these `-100` labels.

The attention mask is different. A `1` marks real text the model can read. A `0`
marks padding added to make a rectangular batch. Padding is masked by position,
so an actual end token still teaches the model even when padding uses the same ID.

### Cross-entropy in simple terms

Suppose the correct next token is `15`. A model assigning probability 0.8 to `15`
receives loss $-\log(0.8)\approx0.223$. Assigning probability 0.1 receives loss
$-\log(0.1)\approx2.303$. A smaller loss means the correct next token surprised it less.

For the response targets $\mathcal R$, we minimize

$$L_{\mathrm{SFT}}=-\frac{1}{|\mathcal R|}\sum_{t\in\mathcal R}\log p_\theta(y_t\mid x,y_{<t}).$$

Position $t$ predicts token $t+1$. The implementation pairs `logits[:, :-1, :]`
with `labels[:, 1:]` exactly once. Accumulation divides by the actual number of
response targets across the group, including a shorter final group.

![Causal prediction alignment](assets/05_qwen38_restart/causal_alignment.png)

![Response-only labels and padding](assets/03_sft_cold_start/response_mask.png)

*An illustrative label mask. Real token IDs are inspected in the next cell.*


In [ ]:
from src.sft import collate_examples

# Two unequal examples make padding visible. Shapes are [batch_size, sequence_length].
preview = collate_examples(train_examples[:2], tokenizer.pad_token_id)
print({name: tuple(tensor.shape) for name, tensor in preview.items()})
# The last prompt position predicts the first response ID after the causal shift.
assert (
    preview["labels"][0, example["prompt_length"]].item()
    == example["input_ids"][example["prompt_length"]]
)
assert (preview["labels"][preview["attention_mask"] == 0] == -100).all()
assert example["labels"][-1] == tokenizer.eos_token_id
print(
    "Prompt tokens",
    example["prompt_length"],
    "response targets including EOS",
    example["target_length"],
)
rows = [
    {
        "sequence_position": i,
        "token_id": token_id,
        "decoded_token": repr(tokenizer.decode([token_id])),
        "label": example["labels"][i],
        "role": "prompt" if i < example["prompt_length"] else "response / EOS",
    }
    for i, token_id in enumerate(example["input_ids"])
]
display(
    pd.DataFrame(rows).iloc[max(0, example["prompt_length"] - 3) : example["prompt_length"] + 8]
)

## 4. Measure the starting student

The shared workflow first measures reference-token NLL and generates answers to
all 64 validation questions. These answer different questions. NLL measures how
well the model predicts worked solutions. Generated numeric matches measure its
own answers. Both are retained before any optimizer update.

Baseline generation receives only the question prompt. Worked solutions never
enter those generation inputs.

## 5. Train for one short epoch

The training loop processes eight responses before each optimizer step. It clears
old gradients, accumulates response-only cross-entropy, checks finite gradients,
clips their norm to 1.0, and updates the student using AdamW. Learning rate starts
with linear warmup and decreases over the fixed 32-update schedule.

Every eight updates, an atomic checkpoint preserves model weights, optimizer,
scheduler, and RNG state. A ten-minute SLURM reserve allows a controlled pause.
The same [shared training runner](../src/opd_runner.py) powers notebook and CPU tests.
The SFT path uses `backward_sft_group`; teacher scoring begins later in 04.

In [ ]:
# Actual training is an explicit opt-in. Inspection stops before loading weights.
if MODE == "train":
    sft_run = run_cold_start(state, RESUME_RUN_DIRECTORY)
    print("Saved run", sft_run["directory"])
    print("Status", sft_run["result"]["status"])
else:
    sft_run = None
    print("Inspection completed. GPU baseline and training measurements are pending.")

## 6. Evaluate what changed

The final student answers the same validation questions under the same greedy
policy. Report strict numeric matching, answer-format compliance, exploratory
prose numeric matching, and token-limit stops separately. Correct numbers alone
cannot verify every reasoning step.

SFT can teach the requested format while arithmetic remains mixed. Compare the
actual before/after answers and include regressions alongside improvements.

In [ ]:
import json

# Display real measurements only after a completed run. A pause keeps its recovery state.
if sft_run and sft_run["result"]["status"] == "completed":
    directory = Path(sft_run["directory"])
    before = json.loads((directory / "before_validation.json").read_text())
    after = json.loads(
        (
            directory
            / f"sft_control_boundary_{len(sft_run['result']['history']):04d}_validation.json"
        ).read_text()
    )
    display(
        pd.DataFrame(
            [
                {
                    "stage": name,
                    "strict_matches": value["scores"]["strict"]["number_correct"],
                    "format_compliance": value["scores"]["strict"]["parse_rate"],
                    "prose_matches": value["scores"]["prose"]["number_correct"],
                    "token_limit_stops": value["scores"]["prose"]["number_token_limit_stops"],
                }
                for name, value in (("before", before), ("after", after))
            ]
        )
    )
    print("Final reference NLL", sft_run["result"]["final_validation_reference_nll"])
else:
    print("No completed new-profile SFT result to display in this kernel.")

## 7. Preserve the checkpoint

The checkpoint holds model weights and tokenizer files. The small report holds
configuration, split identities, measurements, source snapshots, and checksums.
They serve different purposes. On Amarel, 04 can read the completed run directory
directly. Keep it on persistent storage and retain a separate backup.

If interrupted, set `RESUME_RUN_DIRECTORY` to the printed run folder and rerun
from the top. Completed baseline answers and optimizer boundaries are reused only
when experiment identities match. The reserved test remains deferred.

In [ ]:
# Full weights remain separate from the lightweight report. No browser download starts here.
if sft_run:
    print("Run directory", sft_run["directory"])
    print("Checkpoint", sft_run.get("checkpoint"))
    print("Report", sft_run.get("report"))
    print("Use this run directory as SFT_RUN_DIRECTORY in 04.")

## What we achieved, in simple terms

**01** followed one response through probabilities, teacher feedback, gradients,
and an update. **02** established a baseline and separated formatting from numeric
matching. **03** now prepares a fresh supervised starting point for the selected
model and verifies its handoff to 04.

In inspection mode, we checked data and token boundaries. After a completed GPU
run, the student has learned from 256 worked examples and has measured validation
results plus a reload-verified checkpoint. **04 will let the student write its own
responses and learn from the frozen teacher's feedback on those exact prefixes.**

This is a small educational adaptation of GKD's supervised starting point. Its
models, reference data, scale, and optimizer differ from the published experiment.
[GKD Sections 3.1 and 4.3](https://arxiv.org/html/2306.13649v3)